# 🐬 Dolphin (uncensored) di Colab — Ollama + Cloudflare Tunnel

Jalankan model **dolphin-mistral** (uncensored) gratis di Google Colab, expose sebagai API publik.

**Cara pakai:**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Di cell 4, kamu dapat **URL API publik** — simpan URL itu

> ⚠️ Sesi Colab gratis max ~12 jam. Kalau sesi mati, jalankan ulang semua cell (URL tunnel akan berubah).

In [ ]:
# @title 1️⃣ Install Ollama
!nvidia-smi -L 2>/dev/null || echo "⚠️ Tidak ada GPU — model tetap jalan tapi lambat (Runtime → Change runtime type → T4 GPU)"
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -2


In [ ]:
# @title 2️⃣ Start server Ollama
import os, subprocess, time

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
server = subprocess.Popen(["ollama", "serve"],
                          stdout=open("ollama.log", "w"),
                          stderr=subprocess.STDOUT)
time.sleep(5)

status = os.popen("curl -s http://localhost:11434/").read().strip()
print(status)
assert "Ollama" in status, "Server belum siap — jalankan cell ini sekali lagi"


In [ ]:
# @title 3️⃣ Download model (~4GB, tunggu 2–5 menit)
MODEL = "dolphin-mistral"  # @param ["dolphin-mistral", "dolphin3", "llama3.1:8b", "qwen2.5:7b"]

!ollama pull {MODEL}
!ollama list


In [ ]:
# @title 4️⃣ Buka tunnel publik (Cloudflare) → dapatkan URL API 🎉
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://localhost:11434 > tunnel.log 2>&1 &

import time, re

url = None
for _ in range(6):
    time.sleep(5)
    log = open("tunnel.log").read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    if m:
        url = m.group(0)
        break

assert url, "URL belum muncul — jalankan cell ini sekali lagi"
with open("api_url.txt", "w") as f:
    f.write(url)

print("=" * 55)
print("🎉 API KAMU SIAP (simpan URL ini):")
print(url)
print("=" * 55)


In [ ]:
# @title 5️⃣ Tes chat
import requests

r = requests.post(
    f"{url}/v1/chat/completions",
    json={"model": MODEL,
          "messages": [{"role": "user", "content": "Halo! Perkenalkan dirimu dalam 1 kalimat."}]},
    timeout=120,
)
print(r.json()["choices"][0]["message"]["content"])


In [ ]:
# @title 6️⃣ Info pemakaian dari luar Colab
print(f"""
API SIAP DIPAKAI DARI MANA SAJA
--------------------------------
Base URL : {url}/v1
API Key  : ollama  (isi bebas)
Model    : {MODEL}

Contoh dari laptopmu (Python):

    from openai import OpenAI

    client = OpenAI(base_url="{url}/v1", api_key="ollama")
    r = client.chat.completions.create(
        model="{MODEL}",
        messages=[{{"role": "user", "content": "halo"}}],
    )
    print(r.choices[0].message.content)
""")


## Catatan
- 🕐 Sesi Colab gratis **max ~12 jam** — kalau mati, jalankan ulang semua cell (URL berubah)
- 🔒 Endpoint ini **publik tanpa auth** — siapa pun yang tahu URL-nya bisa pakai. Jangan sebarkan URL, dan matikan sesi saat selesai
- 🔄 Ganti model lewat dropdown di cell 3 — daftar model lain: [ollama.com/library](https://ollama.com/library)